In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression # Linear counterpart for classification
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import warnings

# Preprocessing usually remains similar
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Note: statsmodels.api.Logit is used for classification instead of OLS
import statsmodels.api as sm 


In [2]:
df = pd.read_csv('data.csv')
df.drop(columns=['Unnamed: 0'],axis=1, inplace = True)

In [3]:
X = df.drop(columns=['target'],axis=1)
y = df['target']

In [4]:
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalachh,exang,oldpeak,slope,ca,thal
0,63,1,3,145.0,233.0,1.0,0.0,150.0,0.0,2.3,0.0,0.0,1.0
1,37,1,2,130.0,250.0,0.0,1.0,187.0,0.0,3.5,0.0,0.0,2.0
2,41,0,1,130.0,204.0,0.0,0.0,172.0,0.0,1.4,2.0,0.0,2.0
3,56,1,1,120.0,236.0,0.0,1.0,178.0,0.0,0.8,2.0,0.0,2.0
4,57,0,0,120.0,354.0,0.0,1.0,163.0,1.0,0.6,2.0,0.0,2.0


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((481, 13), (121, 13))

In [6]:
scaler= StandardScaler()

In [7]:
data_train =scaler.fit_transform(X_train)
data_test = scaler.transform(X_test)

In [8]:
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    f1 = f1_score(true, predicted, average='weighted')
    precision = precision_score(true, predicted, average='weighted')
    recall = recall_score(true, predicted, average='weighted')
    
    return accuracy, f1, precision, recall


In [9]:
models = {
    "LogisticRegression": LogisticRegression(),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(), 
    "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier(),
    "SVC": SVC()}
model_list = []
accuracy_list = []
accuracy_train = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(data_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(data_train)
    y_test_pred = model.predict(data_test)
    
    # Evaluate Train and Test dataset
    # Note: evaluate_model now returns (accuracy, f1, precision, recall)
    train_accuracy, train_f1, train_precision, train_recall = evaluate_model(y_train, y_train_pred)
    test_accuracy, test_f1, test_precision, test_recall = evaluate_model(y_test, y_test_pred)
    
    model_name = list(models.keys())[i]
    model_list.append(model_name)
    accuracy_list.append(test_accuracy)
    accuracy_train.append(train_accuracy)
    
    print(model_name)
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(train_accuracy))
    print("- F1 Score: {:.4f}".format(train_f1))
    print("- Precision: {:.4f}".format(train_precision))
    print("- Recall: {:.4f}".format(train_recall))
    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Accuracy: {:.4f}".format(test_accuracy))
    print("- F1 Score: {:.4f}".format(test_f1))
    print("- Precision: {:.4f}".format(test_precision))
    print("- Recall: {:.4f}".format(test_recall))
    
    print('='*35)
    print('\n')


LogisticRegression
Model performance for Training set
- Accuracy: 0.6923
- F1 Score: 0.6924
- Precision: 0.6925
- Recall: 0.6923
----------------------------------
Model performance for Test set
- Accuracy: 0.6942
- F1 Score: 0.6940
- Precision: 0.6946
- Recall: 0.6942


K-Neighbors Classifier
Model performance for Training set
- Accuracy: 0.8129
- F1 Score: 0.8127
- Precision: 0.8130
- Recall: 0.8129
----------------------------------
Model performance for Test set
- Accuracy: 0.7025
- F1 Score: 0.7025
- Precision: 0.7025
- Recall: 0.7025


Decision Tree
Model performance for Training set
- Accuracy: 1.0000
- F1 Score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.7355
- F1 Score: 0.7355
- Precision: 0.7359
- Recall: 0.7355


Random Forest Classifier
Model performance for Training set
- Accuracy: 1.0000
- F1 Score: 1.0000
- Precision: 1.0000
- Recall: 1.0000
----------------------------------
Model performan

In [10]:
pd.DataFrame(list(zip(model_list, accuracy_list, accuracy_train)), columns=['Model Name', 'Accuracy', 'Train Accuracy']).sort_values(by=["Accuracy"], ascending=False)


,Model Name,Accuracy,Train Accuracy
5,CatBoosting Classifier,0.793388,0.989605
3,Random Forest Classifier,0.785124,1.000000
7,SVC,0.760331,0.879418
4,XGBClassifier,0.752066,1.000000
2,Decision Tree,0.735537,1.000000
6,AdaBoost Classifier,0.735537,0.825364
1,K-Neighbors Classifier,0.702479,0.812890
0,LogisticRegression,0.694215,0.692308


In [12]:
# hyperparameter with hyperpara

In [24]:
param_grid = {
    'C': [0.1, 1, 10,25,50 ,100],
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'gamma': ['scale', 'auto'],
    
}
model = SVC()
search = GridSearchCV(model, param_grid, cv=5, scoring ="accuracy")

In [27]:
search.fit(data_train,y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10, 25, 50, 100],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring='accuracy')

In [28]:
scv_test = search.predict(data_test)

In [29]:
scv_train = search.predict(data_train)

In [31]:
 train_accuracy, train_f1, train_precision, train_recall = evaluate_model(y_train, scv_train)
test_accuracy, test_f1, test_precision, test_recall = evaluate_model(y_test, scv_test)
    

In [32]:
print(train_accuracy, train_f1, train_precision, train_recall)

0.9708939708939709 0.9708765702215485 0.9711623688718951 0.9708939708939709


In [33]:
print(test_accuracy, test_f1, test_precision, test_recall)

0.7851239669421488 0.7851239669421488 0.7851239669421488 0.7851239669421488


In [ ]:
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [100, 200, 500],
    'l2_leaf_reg': [1, 3, 5, 7]
    
}
model = CatBoostClassifier()
a_search = GridSearchCV(model, param_grid, cv=5, scoring ="accuracy")

In [ ]:
a_search.fit(data_train,y_train)

0:	learn: 0.6897302	total: 1.03ms	remaining: 102ms
1:	learn: 0.6862346	total: 2ms	remaining: 98ms
2:	learn: 0.6835699	total: 2.45ms	remaining: 79.2ms
3:	learn: 0.6803470	total: 3.04ms	remaining: 72.9ms
4:	learn: 0.6770088	total: 3.82ms	remaining: 72.6ms
5:	learn: 0.6740170	total: 4.69ms	remaining: 73.5ms
6:	learn: 0.6724291	total: 5.53ms	remaining: 73.5ms
7:	learn: 0.6696523	total: 6.17ms	remaining: 70.9ms
8:	learn: 0.6664457	total: 6.94ms	remaining: 70.2ms
9:	learn: 0.6633740	total: 7.97ms	remaining: 71.8ms
10:	learn: 0.6608302	total: 8.98ms	remaining: 72.6ms
11:	learn: 0.6588721	total: 9.84ms	remaining: 72.2ms
12:	learn: 0.6559720	total: 11.1ms	remaining: 74.1ms
13:	learn: 0.6539236	total: 11.8ms	remaining: 72.5ms
14:	learn: 0.6525546	total: 12.5ms	remaining: 70.6ms
15:	learn: 0.6497392	total: 14ms	remaining: 73.5ms
16:	learn: 0.6473029	total: 15.1ms	remaining: 73.8ms
17:	learn: 0.6456199	total: 15.8ms	remaining: 71.8ms
18:	learn: 0.6438984	total: 16.6ms	remaining: 70.8ms
19:	learn: 

In [43]:
cat_test = a_search.predict(data_test)

In [44]:
cat_train = a_search.predict(data_train)

In [45]:
train_accuracy, train_f1, train_precision, train_recall = evaluate_model(y_train, cat_train)
test_accuracy, test_f1, test_precision, test_recall = evaluate_model(y_test, cat_test)

In [46]:
print(train_accuracy, train_f1, train_precision, train_recall)

0.8794178794178794 0.8791413917873061 0.8808282883754582 0.8794178794178794


In [47]:
print(test_accuracy, test_f1, test_precision, test_recall)

0.768595041322314 0.7685634286230412 0.7689520082780186 0.768595041322314


In [51]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1, 10] 
    
}
model = AdaBoostClassifier()
b_search = GridSearchCV(model, param_grid, cv=5, scoring ="accuracy")

In [52]:
b_search.fit(data_train,y_train)

GridSearchCV(cv=5, estimator=AdaBoostClassifier(),
             param_grid={'learning_rate': [0.01, 0.1, 1, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

In [53]:
ada_test = b_search.predict(data_test)

In [54]:
ada_train = b_search.predict(data_train)

In [56]:
train_accuracy, train_f1, train_precision, train_recall = evaluate_model(y_train, ada_train)
test_accuracy, test_f1, test_precision, test_recall = evaluate_model(y_test, ada_test)

In [57]:
print(train_accuracy, train_f1, train_precision, train_recall)

0.8087318087318087 0.8082932421453821 0.8096581341864362 0.8087318087318087


In [58]:
print(test_accuracy, test_f1, test_precision, test_recall)

0.7355371900826446 0.7353202987677862 0.7367152385094969 0.7355371900826446


In [59]:
param_grid = {
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
    
}
model = DecisionTreeClassifier()
dt_search = GridSearchCV(model, param_grid, cv=5, scoring ="accuracy")

In [60]:
dt_search.fit(data_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 5, 10], 'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [61]:
decision_test = dt_search.predict(data_test)

In [62]:
decision_train = dt_search.predict(data_train)

In [63]:
train_accuracy, train_f1, train_precision, train_recall = evaluate_model(y_train, decision_train)
test_accuracy, test_f1, test_precision, test_recall = evaluate_model(y_test, decision_test)


In [64]:
print(train_accuracy, train_f1, train_precision, train_recall )

0.8627858627858628 0.8612937764596533 0.8734743926279466 0.8627858627858628


In [65]:
print(test_accuracy, test_f1, test_precision, test_recall)

0.7272727272727273 0.7267125206187172 0.7286501377410469 0.7272727272727273


In [67]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
    
}
model = RandomForestClassifier()
rt_search = GridSearchCV(model, param_grid, cv=5, scoring ="accuracy")

In [68]:
rt_search.fit(data_train,y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'max_depth': [3, 5, 10],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='accuracy')

In [69]:
random_test = rt_search.predict(data_test)

In [70]:
random_train = rt_search.predict(data_train)

In [71]:
train_accuracy, train_f1, train_precision, train_recall = evaluate_model(y_train, random_train)
test_accuracy, test_f1, test_precision, test_recall = evaluate_model(y_test, random_test)

In [72]:
print(train_accuracy, train_f1, train_precision, train_recall)

0.9126819126819127 0.9124428783934467 0.9147882039827855 0.9126819126819127


In [73]:
print(test_accuracy, test_f1, test_precision, test_recall)

0.8016528925619835 0.8016257959626067 0.8020369709320452 0.8016528925619835


In [74]:
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
    
}
model = XGBClassifier()
xgb_search = GridSearchCV(model, param_grid, cv=5, scoring ="accuracy")

In [75]:
xgb_search.fit(data_train,y_train)

GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=Non...
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [3, 5], 'n_estimators': [100, 200],
                         'subsample': [0.8, 1.0]},
             scoring='accuracy')

In [76]:
xgb_test = xgb_search.predict(data_test)

In [77]:
xgb_train = xgb_search.predict(data_train)

In [78]:
train_accuracy, train_f1, train_precision, train_recall = evaluate_model(y_train, xgb_train)
test_accuracy, test_f1, test_precision, test_recall = evaluate_model(y_test, xgb_test)

In [79]:
print(train_accuracy, train_f1, train_precision, train_recall)

0.8669438669438669 0.8667430013603882 0.8676277434898124 0.8669438669438669


In [80]:
print(test_accuracy, test_f1, test_precision, test_recall )

0.7851239669421488 0.7846829241813567 0.7881292261457552 0.7851239669421488


In [81]:
print("Best Params:", xgb_search.best_params_)
print("Best Score:", xgb_search.best_score_)

Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}
Best Score: 0.8108247422680412
